# Session 4: RAG End to End

**RAG = Retrieval-Augmented Generation.** Retrieve the relevant pieces of *your* documents, then let the LLM generate an answer *from those pieces*.


| Part | What we do |
|---|---|
| 0 | Setup |
| 1 | Why RAG: the LLM alone does not know our documents |
| 2 | Load PDFs and look at what was really extracted |
| 3 | Clean and attach useful metadata |
| 4 | Split into chunks (and check the embedding token limit) |
| 5 | Embed and store in Chroma |
| 6 | Retrieve: similarity, MMR, score threshold, filters |
| 7 | Generate grounded answers with citations |

**Documents** (`docs/`): four policy PDFs of the fictional Sahyadri Digital Bank, including a **superseded 2024 loan policy** that contradicts the 2026 one.

## Setup: run this cell first

Choose the provider and model in the cell below. The API key is **not** written here: it stays in the `.env` file in this folder, for example:

```
OPENROUTER_API_KEY=sk-or-v1-...
```

In [1]:
# Setup
# "gemini" (free) | "groq" (free) | "openrouter" (free :free models) | "openai" (paid)
PROVIDER = "gemini"
MODEL = ""          # leave empty to use the default model below

import os
from dotenv import load_dotenv

load_dotenv(override=True)

PROVIDERS = {
    "gemini": {
        "key_name": "GOOGLE_API_KEY",
        "model": "gemini-3.1-flash-lite",
        "base_url": "https://generativelanguage.googleapis.com/v1beta/openai/",
    },
    "groq": {
        "key_name": "GROQ_API_KEY",
        "model": "openai/gpt-oss-20b",          # bigger: openai/gpt-oss-120b
        "base_url": "https://api.groq.com/openai/v1",
    },
    "openrouter": {
        "key_name": "OPENROUTER_API_KEY",
        "model": "openai/gpt-oss-20b:free",     # or "openrouter/free"
        "base_url": "https://openrouter.ai/api/v1",
    },
    "openai": {
        "key_name": "OPENAI_API_KEY",
        "model": "gpt-4.1-mini",
        "base_url": "https://api.openai.com/v1",
    },
}

if PROVIDER not in PROVIDERS:
    raise ValueError(f"PROVIDER must be one of {list(PROVIDERS)}")

KEY_NAME = PROVIDERS[PROVIDER]["key_name"]
MODEL = MODEL or PROVIDERS[PROVIDER]["model"]
BASE_URL = PROVIDERS[PROVIDER]["base_url"]

API_KEY = os.getenv(KEY_NAME)
if not API_KEY:
    try:
        from google.colab import userdata   # Colab secrets
        API_KEY = userdata.get(KEY_NAME)
    except Exception:
        pass
if not API_KEY:
    raise ValueError(f"{KEY_NAME} is missing. Add it to .env (or Colab secrets).")


def get_chat_model(temperature=0, **kwargs):
    """LangChain chat model for the chosen provider."""
    if PROVIDER == "gemini":
        from langchain_google_genai import ChatGoogleGenerativeAI
        return ChatGoogleGenerativeAI(model=MODEL, api_key=API_KEY, temperature=temperature, **kwargs)
    elif PROVIDER == "groq":
        from langchain_groq import ChatGroq
        return ChatGroq(model=MODEL, api_key=API_KEY, temperature=temperature, **kwargs)
    else:
        from langchain_openai import ChatOpenAI
        return ChatOpenAI(model=MODEL, api_key=API_KEY, base_url=BASE_URL, temperature=temperature, **kwargs)


def get_openai_client():
    """Plain OpenAI SDK client pointed at the chosen provider. Returns (client, model)."""
    from openai import OpenAI
    return OpenAI(api_key=API_KEY, base_url=BASE_URL), MODEL


def show_config():
    print(f"provider : {PROVIDER}")
    print(f"model    : {MODEL}")
    print(f"key      : {KEY_NAME} = {API_KEY[:6]}...{API_KEY[-2:]} ({len(API_KEY)} chars)")
    print(f"endpoint : {BASE_URL}")


show_config()

provider : gemini
model    : gemini-3.1-flash-lite
key      : GOOGLE_API_KEY = AIzaSy...co (39 chars)
endpoint : https://generativelanguage.googleapis.com/v1beta/openai/


## Part 0: Setup

In [2]:
import os
import re
import glob

import numpy as np
import pandas as pd

from importlib.metadata import version

for pkg in ["langchain-core", "langchain-text-splitters", "langchain-chroma", "pypdf", "chromadb"]:
    print(f"{pkg:25s} {version(pkg)}")

llm = get_chat_model(temperature=0)

langchain-core            1.6.7
langchain-text-splitters  1.1.3
langchain-chroma          1.1.0
pypdf                     6.19.0
chromadb                  1.5.9


/home/aditya/AGENTIC_AI_KBT_OCT_2026/python==3.12/lib/python3.14/site-packages/langchain_core/utils/pydantic.py:42: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


## Part 1: Why RAG

Ask the LLM directly about our bank's policy:

In [3]:
question = "What is the foreclosure charge on a personal loan at Sahyadri Digital Bank?"
print(llm.invoke(question).content)

[{'type': 'text', 'text': 'To provide you with the most accurate information, it is important to clarify that **"Sahyadri Digital Bank" is not a recognized or regulated commercial bank** in India.\n\nIf you are referring to a specific cooperative society, a digital lending app, or a local credit union, please be aware of the following:\n\n### 1. Verify the Entity\n*   **Check the RBI Website:** Before sharing personal details or making payments, check the [Reserve Bank of India (RBI) website](https://www.rbi.org.in/) to see if the entity is a licensed bank or a registered Non-Banking Financial Company (NBFC).\n*   **Beware of Fraud:** Many fraudulent apps use names that sound like legitimate banks to trick users into paying "foreclosure fees" or "processing fees" to release loans that do not exist. **Never pay an upfront fee to receive a loan.**\n\n### 2. Standard Foreclosure Charges (Industry Norms)\nIf you are dealing with a legitimate, RBI-regulated lender in India, foreclosure (pre

The model has never seen this bank's documents. It either says it does not know, or worse, gives a **plausible-sounding guess** based on typical Indian bank charges. Both are useless to a customer.

Options to fix this:
- **Fine-tune** on the documents: expensive, slow to update, and does not reliably store facts (Session 1).
- **Paste all documents** into the prompt: works for 2 pages, not for 2,000; costly per question.
- **RAG**: find the 3 to 5 most relevant chunks and send only those. Cheap, easy to update (just re-index a document), and answers can **cite their source**.

```
INDEXING (once per document)          ANSWERING (every question)
load -> clean -> split -> embed       question -> retrieve chunks -> prompt -> LLM -> answer + sources
     -> store in vector DB
```

## Part 2: Load PDFs

A **document loader** just reads a file and returns `Document` objects (text + metadata). For PDFs we write our own with `pypdf`: one `Document` per page. It is only a few lines, and nothing is hidden.

(Older tutorials use `PyPDFLoader` from `langchain_community`. That package is being sunset by LangChain, so we avoid depending on it. Internally it does the same as the function below.)

Always look at what was actually extracted before doing anything else - most RAG problems start here.

In [4]:
from pypdf import PdfReader
from langchain_core.documents import Document

def load_pdf(path):
    reader = PdfReader(path)
    docs = []
    for index, page in enumerate(reader.pages):
        docs.append(Document(
            page_content=page.extract_text() or "",
            metadata={"source": path, "page": index, "total_pages": len(reader.pages)},
        ))
    return docs

pdf_files = sorted(glob.glob("docs/*.pdf"))
print(pdf_files)

pages = []
for path in pdf_files:
    loaded = load_pdf(path)
    print(f"{os.path.basename(path):35s} {len(loaded)} page(s)")
    pages.extend(loaded)

print("\ntotal pages:", len(pages))

['docs/Credit_Card_Terms_2026.pdf', 'docs/Grievance_Redressal_Policy.pdf', 'docs/Personal_Loan_Policy_2024.pdf', 'docs/Personal_Loan_Policy_2026.pdf']
Credit_Card_Terms_2026.pdf          2 page(s)
Grievance_Redressal_Policy.pdf      1 page(s)
Personal_Loan_Policy_2024.pdf       1 page(s)
Personal_Loan_Policy_2026.pdf       2 page(s)

total pages: 6


Metadata of one page. `page` is the **0-based** index from Python's `enumerate` (the first page is `0`). Humans count from 1, so we convert it before citing pages.

In [11]:
print(pages[1].metadata)

{'source': 'docs/Credit_Card_Terms_2026.pdf', 'page': 1, 'total_pages': 2}


Now the text. Look for three kinds of noise:
1. **Header and footer** lines repeated on every page.
2. **Tables** flattened: each cell on its own line, so a row's label and value are separated.
3. **Line breaks** in the middle of sentences.

In [12]:
page = next(p for p in pages if "Fees and charges" in p.page_content)
print(page.page_content[:1100])

Sahyadri Digital Bank | Personal Loan Policy 2024 | Version 2.1 | SUPERSEDED
Page 1
Sahyadri Digital Bank Ltd. | For customer information
Personal Loan Policy 2024 (superseded)
Note: this policy was replaced by the Personal Loan Policy 2026 for all loans sanctioned on or
after 1 April 2026.
1. Scope
This policy applies to personal loans sanctioned between 1 April 2024 and 31 March 2026.
2. Interest rates
Interest rates start at 11.5% per year for applicants with a credit score of 780 and above, and go up to
16% per year. Rates are fixed for the full tenure.
3. Fees and charges
 Charge
Amount
Processing fee
2% of the loan amount, minimum Rs 2,500
Part-prepayment
2% of the amount prepaid
Foreclosure (full prepayment)
4% of the outstanding principal; not allowed before 12 EMIs
Late payment fee
Rs 600 per missed EMI
4. Tenure
Tenure ranges from 12 to 48 months. The loan amount ranges from Rs 50,000 to Rs 15 lakh.



The table survived as alternating lines (`Foreclosure (full prepayment)` then `2% of the outstanding principal...`). That is readable for an LLM as long as the label and value stay in the **same chunk**. Keep this in mind for Part 4.

Quick size check per page:

In [13]:
for p in pages:
    print(f"{os.path.basename(p.metadata['source']):35s} page {p.metadata['page'] + 1}: {len(p.page_content):5d} characters")

Credit_Card_Terms_2026.pdf          page 1:  1589 characters
Credit_Card_Terms_2026.pdf          page 2:  1288 characters
Grievance_Redressal_Policy.pdf      page 1:  1640 characters
Personal_Loan_Policy_2024.pdf       page 1:   923 characters
Personal_Loan_Policy_2026.pdf       page 1:  2193 characters
Personal_Loan_Policy_2026.pdf       page 2:  2250 characters


## Part 3: Clean and add metadata

### Remove repeated header and footer lines

They appear on every page, so they end up in every chunk: they add noise to every embedding and waste tokens. A simple rule-based cleaner is enough here. (For other documents, look at the raw text first and write rules that match them.)

In [14]:
NOISE_PATTERNS = [
    r"^Sahyadri Digital Bank \|.*$",        # running header
    r"^Sahyadri Digital Bank Ltd\. \|.*$",  # footer
    r"^Page \d+$",                           # page number
]

def clean_text(text):
    lines = text.splitlines()
    kept = [line for line in lines if not any(re.match(p, line.strip()) for p in NOISE_PATTERNS)]
    return "\n".join(kept).strip()

before = pages[1].page_content
after = clean_text(before)
print("BEFORE:", repr(before[:160]))
print("AFTER :", repr(after[:160]))
print(f"\n{len(before)} -> {len(after)} characters")

BEFORE: 'Sahyadri Digital Bank | Credit Card Most Important Terms and Conditions\nPage 2\nSahyadri Digital Bank Ltd. | For customer information\n5. Lost or stolen card\nRepo'
AFTER : '5. Lost or stolen card\nReport a lost or stolen card immediately through the app or by calling 1800-000-1111. The card is\nblocked at once. The cardholder has zer'

1288 -> 1154 characters


### Keep only useful metadata

Metadata is what lets us **filter** (only current policies), **cite** (file and page) and **debug** (where did this chunk come from).

In a company this information usually comes from a document register. Here we write it by hand. Note `status`: the 2024 loan policy is **superseded**.

Chroma accepts only simple metadata values (text, numbers, booleans) - no lists or `None`.

In [15]:
DOC_REGISTER = {
    "Personal_Loan_Policy_2026.pdf": {"doc_type": "loan_policy", "year": 2026, "status": "current"},
    "Personal_Loan_Policy_2024.pdf": {"doc_type": "loan_policy", "year": 2024, "status": "superseded"},
    "Credit_Card_Terms_2026.pdf":    {"doc_type": "card_terms", "year": 2026, "status": "current"},
    "Grievance_Redressal_Policy.pdf": {"doc_type": "grievance", "year": 2026, "status": "current"},
}

clean_pages = []
for p in pages:
    file_name = os.path.basename(p.metadata["source"])
    meta = {"source": file_name, "page": int(p.metadata["page"]) + 1}   # human page number
    meta.update(DOC_REGISTER[file_name])
    clean_pages.append(Document(page_content=clean_text(p.page_content), metadata=meta))

print(len(clean_pages), "clean pages")
print(clean_pages[0].metadata)

6 clean pages
{'source': 'Credit_Card_Terms_2026.pdf', 'page': 1, 'doc_type': 'card_terms', 'year': 2026, 'status': 'current'}


## Part 4: Split into chunks

Why not embed whole pages?

1. **The embedding model has a limit.** `all-MiniLM-L6-v2` reads at most **256 tokens**; anything longer is silently cut off and never searchable.
2. **Precision.** One vector for a whole page mixes many topics; a question about foreclosure has to compete with eligibility and documents on the same page.
3. **Cost.** We send only the relevant chunks to the LLM, not whole pages.

Too small is also bad: a chunk with a table label but not its value is useless.

### RecursiveCharacterTextSplitter

It tries to split on paragraph breaks first (`\n\n`), then line breaks (`\n`), then spaces, then characters - so it cuts at the most natural place that keeps chunks under `chunk_size` characters. `chunk_overlap` repeats the end of one chunk at the start of the next, so a fact on a boundary appears whole in at least one chunk.

See it on a small text with tiny sizes:

In [16]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

sample = ("Foreclosure is not allowed in the first six months of the loan. "
          "After six EMIs the customer may close the loan at any time by paying the outstanding principal, "
          "interest up to the date of closure, and the foreclosure charge.")

demo_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=30)
for i, chunk in enumerate(demo_splitter.split_text(sample)):
    print(f"chunk {i} ({len(chunk)} chars): {chunk}")

chunk 0 (95 chars): Foreclosure is not allowed in the first six months of the loan. After six EMIs the customer may
chunk 1 (98 chars): six EMIs the customer may close the loan at any time by paying the outstanding principal, interest
chunk 2 (74 chars): principal, interest up to the date of closure, and the foreclosure charge.


Notice the repeated words at the start of each chunk: that is the overlap.

### How big is too big? Count tokens, not characters

The embedding model's own tokenizer tells us the real length. We test three chunk sizes on our pages.

You may see a warning like *"Token indices sequence length is longer than the specified maximum sequence length (321 > 256)"*. That warning **is** the problem we are measuring: those chunks would be cut off when embedded.

In [17]:
from sentence_transformers import SentenceTransformer

EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
embed_tokenizer = SentenceTransformer(EMBED_MODEL).tokenizer
EMBED_MAX_TOKENS = 256

def n_tokens(text):
    return len(embed_tokenizer(text)["input_ids"])

rows = []
for size in [300, 800, 1500]:
    splitter = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=size // 7)
    chunks = splitter.split_documents(clean_pages)
    tokens = [n_tokens(c.page_content) for c in chunks]
    rows.append({
        "chunk_size (chars)": size,
        "chunks": len(chunks),
        "avg tokens": int(np.mean(tokens)),
        "max tokens": max(tokens),
        "chunks over 256 tokens": sum(t > EMBED_MAX_TOKENS for t in tokens),
    })

pd.DataFrame(rows)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (343 > 256). Running this sequence through the model will result in indexing errors


,chunk_size (chars),chunks,avg tokens,max tokens,chunks over 256 tokens
0,300,39,55,80,0
1,800,15,145,188,0
2,1500,9,237,343,4


At 1500 characters, chunks exceed 256 tokens: the end of each such chunk is **invisible to search**. At 300, facts get scattered across many tiny chunks. **800 characters with 120 overlap** keeps every chunk under the limit while holding a full table section or paragraph together.

There is no universal best size. It depends on the embedding model's limit and on your documents; Part 9 shows how to measure.

In [18]:
splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=120, add_start_index=True)
chunks = splitter.split_documents(clean_pages)

print(len(chunks), "chunks")
print(chunks[5].metadata)
print()
print(chunks[5].page_content)

15 chunks
{'source': 'Grievance_Redressal_Policy.pdf', 'page': 1, 'doc_type': 'grievance', 'year': 2026, 'status': 'current', 'start_index': 630}

Banking Ombudsman (RBI)
If no reply within 30 days or the customer is not satisfied with the final response
Customers may approach the RBI Ombudsman through the Complaint Management System website
within one year of receiving the bank's final response, or one year and 30 days from the date of the
complaint if no reply was received.
3. Fraud and unauthorised transactions
Complaints about unauthorised electronic transactions are treated as priority. If reported within 3
working days, the customer has zero liability and the amount is credited as a shadow reversal within
10 working days while the bank investigates. The investigation must be completed within 90 days.
4. Compensation
If a failed ATM or UPI transaction is not reversed within 5 days, the bank pays compensation of Rs


Every chunk **inherited** its page's metadata (source, page, status) and gained `start_index` (character position on the page). That is how a chunk knows where it came from.

## Part 5: Embed and store

Same tools as Session 3. Two production habits:

- **Deterministic ids** built from source, page and position. Re-indexing the same document then *updates* chunks instead of duplicating them.
- **Reset by deleting the collection**, not the folder (safe on Windows).

In [19]:
import chromadb
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

DB_DIR = "chroma_db"
COLLECTION = "bank_policies"

client = chromadb.PersistentClient(path=DB_DIR)
if COLLECTION in [c.name for c in client.list_collections()]:
    client.delete_collection(COLLECTION)

embeddings = HuggingFaceEmbeddings(model=EMBED_MODEL, encode_kwargs={"normalize_embeddings": True})

vectorstore = Chroma(
    client=client,
    collection_name=COLLECTION,
    embedding_function=embeddings,
    collection_configuration={"hnsw": {"space": "cosine"}},
)

ids = [f"{c.metadata['source']}-p{c.metadata['page']}-{c.metadata['start_index']}" for c in chunks]
vectorstore.add_documents(chunks, ids=ids)

print("stored chunks:", vectorstore._collection.count())
print("example id   :", ids[0])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

stored chunks: 15
example id   : Credit_Card_Terms_2026.pdf-p1-0


In [21]:
vectorstore.get();

In [22]:
dir(vectorstore)

['_Chroma__ensure_collection',
 '_Chroma__query_collection',
 '_LANGCHAIN_DEFAULT_COLLECTION_NAME',
 '__abstractmethods__',
 '__annotate_func__',
 '__annotations_cache__',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__firstlineno__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__slots__',
 '__static_attributes__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_abc_impl',
 '_asimilarity_search_with_relevance_scores',
 '_chroma_collection',
 '_client',
 '_collection',
 '_collection_configuration',
 '_collection_metadata',
 '_collection_name',
 '_cosine_relevance_score_fn',
 '_embedding_function',
 '_euclidean_relevance_score_fn',
 '_get_retriever_tags',
 '_max_inner_product_relevance_score_fn',
 '_select_relevance_score_fn',
 '_simil

In [25]:
vectorstore.similarity_search("policy");

In [29]:
vectorstore.similarity_search_with_score('loan default', k = 1)

[(Document(id='Personal_Loan_Policy_2024.pdf-p1-0', metadata={'source': 'Personal_Loan_Policy_2024.pdf', 'page': 1, 'doc_type': 'loan_policy', 'start_index': 0, 'status': 'superseded', 'year': 2024}, page_content='Personal Loan Policy 2024 (superseded)\nNote: this policy was replaced by the Personal Loan Policy 2026 for all loans sanctioned on or\nafter 1 April 2026.\n1. Scope\nThis policy applies to personal loans sanctioned between 1 April 2024 and 31 March 2026.\n2. Interest rates\nInterest rates start at 11.5% per year for applicants with a credit score of 780 and above, and go up to\n16% per year. Rates are fixed for the full tenure.\n3. Fees and charges\n Charge\nAmount\nProcessing fee\n2% of the loan amount, minimum Rs 2,500\nPart-prepayment\n2% of the amount prepaid\nForeclosure (full prepayment)\n4% of the outstanding principal; not allowed before 12 EMIs\nLate payment fee\nRs 600 per missed EMI\n4. Tenure\nTenure ranges from 12 to 48 months. The loan amount ranges from Rs 50,

## Part 6: Retrieve

**Rule one of RAG debugging: look at what was retrieved before blaming the LLM.** If the right chunk is not retrieved, no prompt can save the answer.

A helper that prints retrieved chunks with source, page and score:

In [30]:
def show(results, width=110):
    for item in results:
        doc, score = item if isinstance(item, tuple) else (item, None)
        m = doc.metadata
        head = f"[{m['source']} p{m['page']} {m['status']}]"
        score_txt = f" relevance={score:.3f}" if score is not None else ""
        text = " ".join(doc.page_content.split())
        print(f"{head}{score_txt}\n    {text[:width]}...")

show(vectorstore.similarity_search_with_relevance_scores("Can I close my personal loan early and what does it cost?", k=4))

[Personal_Loan_Policy_2024.pdf p1 superseded] relevance=0.553
    Personal Loan Policy 2024 (superseded) Note: this policy was replaced by the Personal Loan Policy 2026 for all...
[Personal_Loan_Policy_2026.pdf p1 current] relevance=0.454
    Personal Loan Policy (effective 1 April 2026) 1. Purpose and scope This policy describes how Sahyadri Digital ...
[Personal_Loan_Policy_2026.pdf p2 current] relevance=0.446
    5. Fees and charges All fees below are exclusive of GST at 18%. Charge Amount Processing fee 1.5% of the loan ...
[Personal_Loan_Policy_2026.pdf p1 current] relevance=0.347
    3. Loan amount and tenure The loan amount ranges from Rs 50,000 to Rs 25 lakh. The maximum amount for an appli...


Look at the sources: typically both the **2026** and the **superseded 2024** policy come back, with different foreclosure rules (2% vs 4%). Retrieval found what we asked for; it has no idea which document is valid. We handle this in Part 8.

### Retriever options

| search_type | Behaviour | Use when |
|---|---|---|
| `similarity` | top k most similar | default |
| `mmr` | Maximal Marginal Relevance: relevant **and** different from each other | top results are near-duplicates (overlapping chunks, repeated text) |
| `similarity_score_threshold` | only results above a minimum relevance | you prefer "nothing found" over weak matches |

MMR first fetches `fetch_k` candidates, then picks `k` of them, balancing relevance against similarity to already-picked chunks (`lambda_mult`: 1 = pure relevance, 0 = maximum diversity).

In [31]:
query = "fees and charges"

sim_retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
mmr_retriever = vectorstore.as_retriever(search_type="mmr", search_kwargs={"k": 4, "fetch_k": 20, "lambda_mult": 0.5})

print("SIMILARITY:")
show(sim_retriever.invoke(query), width=80)
print("\nMMR:")
show(mmr_retriever.invoke(query), width=80)

SIMILARITY:
[Credit_Card_Terms_2026.pdf p1 current]
    an interest-free period of 20 to 50 days depending on the purchase date. 3. Inte...
[Personal_Loan_Policy_2026.pdf p2 current]
    the customer chooses it, the premium is shown separately and can be paid upfront...
[Credit_Card_Terms_2026.pdf p1 current]
    Credit Card Terms and Conditions (2026) 1. Card variants and annual fees Card Jo...
[Personal_Loan_Policy_2026.pdf p2 current]
    5. Fees and charges All fees below are exclusive of GST at 18%. Charge Amount Pr...

MMR:
[Credit_Card_Terms_2026.pdf p1 current]
    an interest-free period of 20 to 50 days depending on the purchase date. 3. Inte...
[Personal_Loan_Policy_2026.pdf p2 current]
    the customer chooses it, the premium is shown separately and can be paid upfront...
[Personal_Loan_Policy_2026.pdf p2 current]
    foreclosure charge. 6. Documents required Salaried applicants: PAN card, Aadhaar...
[Grievance_Redressal_Policy.pdf p1 current]
    Customer Grievance Redress

MMR usually spreads results across more documents or sections. Neither is always better; check on your own questions.

**Score threshold:** an out-of-scope question should retrieve nothing rather than random chunks.

In [32]:
threshold_retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"k": 4, "score_threshold": 0.35},
)

for q in ["What is the late fee on a credit card?", "Who won the cricket world cup?"]:
    found = threshold_retriever.invoke(q)
    print(f"{q} -> {len(found)} chunk(s) above threshold")

No relevant docs were retrieved using the relevance score threshold 0.35


What is the late fee on a credit card? -> 4 chunk(s) above threshold
Who won the cricket world cup? -> 0 chunk(s) above threshold


The right threshold depends on the embedding model; 0.35 is a starting point for MiniLM with cosine. Too high and real questions get nothing.

**Metadata filter:** search only current documents.

In [33]:
current_retriever = vectorstore.as_retriever(search_kwargs={"k": 4, "filter": {"status": "current"}})
show(current_retriever.invoke("Can I close my personal loan early and what does it cost?"), width=80)

[Personal_Loan_Policy_2026.pdf p1 current]
    Personal Loan Policy (effective 1 April 2026) 1. Purpose and scope This policy d...
[Personal_Loan_Policy_2026.pdf p2 current]
    5. Fees and charges All fees below are exclusive of GST at 18%. Charge Amount Pr...
[Personal_Loan_Policy_2026.pdf p1 current]
    3. Loan amount and tenure The loan amount ranges from Rs 50,000 to Rs 25 lakh. T...
[Personal_Loan_Policy_2026.pdf p2 current]
    offer. Other applications are decided within 3 working days of receiving complet...


## Part 7: Generate grounded answers with citations

### Formatting the context

Each chunk is labelled with its source, page and status, so the LLM can cite and can tell current from superseded rules.

In [34]:
def format_docs(docs):
    blocks = []
    for d in docs:
        m = d.metadata
        blocks.append(f"[source: {m['source']}, page {m['page']}, status: {m['status']}]\n{d.page_content}")
    return "\n\n---\n\n".join(blocks)

print(format_docs(current_retriever.invoke("late payment fee on personal loan"))[:700])

[source: Personal_Loan_Policy_2026.pdf, page 2, status: current]
offer. Other applications are decided within 3 working days of receiving complete documents. If an
application is rejected, the customer is informed of the main reason in writing.
8. Missed payments
If an EMI is not paid on the due date, the late payment fee and penal charge apply, and the bank
reports the delay to credit bureaus. After 90 days of non-payment the account is classified as a
non-performing asset and recovery action may begin. Customers facing temporary financial difficulty
should contact the bank before the due date to discuss a revised repayment schedule.
9. Insurance
Loan protection insurance is optional. The b


### The grounded prompt

Every rule in the system message targets a specific failure:

| Rule | Prevents |
|---|---|
| answer only from the context | made-up facts from training data |
| cite source and page | unverifiable answers |
| prefer `current` over `superseded` | old rules being quoted |
| say you could not find it | guessing when retrieval missed |

In [35]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda

SYSTEM_PROMPT = (
    "You are a customer support assistant for Sahyadri Digital Bank.\n"
    "Rules:\n"
    "1. Answer ONLY from the context below. Do not use outside knowledge.\n"
    "2. Cite every fact as [file, page].\n"
    "3. If sources disagree, follow the one with status 'current'. Mention a superseded rule only if it still applies to the customer.\n"
    "4. If the context does not contain the answer, reply exactly: I could not find this in the bank's documents.\n"
    "5. Be concise: at most 5 sentences."
)

rag_prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "Context:\n{context}\n\nQuestion: {question}"),
])

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

rag_chain = (
    RunnableParallel({"context": retriever | format_docs, "question": RunnablePassthrough()})
    | rag_prompt
    | llm
    | StrOutputParser()
)

print(rag_chain.invoke("What is the foreclosure charge on a personal loan?"))

The foreclosure charge is 2% of the outstanding principal [Personal_Loan_Policy_2026.pdf, page 2]. This is only permitted after 6 EMIs have been paid [Personal_Loan_Policy_2026.pdf, page 2].


In [36]:
print(rag_chain.invoke("Who won the IPL"))

I could not find this in the bank's documents.


### Returning the sources too

An application should show the user **which documents** the answer came from, not just the text. Keep the retrieved documents in the output with `.assign()`: it runs a step and **adds** its result to the dict instead of replacing it.

In [37]:
answer_step = (
    RunnableLambda(lambda x: {"context": format_docs(x["docs"]), "question": x["question"]})
    | rag_prompt
    | llm
    | StrOutputParser()
)

rag_with_sources = RunnableParallel({"docs": retriever, "question": RunnablePassthrough()}).assign(answer=answer_step)

result = rag_with_sources.invoke("I lost my credit card two days ago and someone used it. Do I have to pay?")

print("keys:", list(result.keys()))
print("\nANSWER:\n" + result["answer"])
print("\nSOURCES:")
for d in result["docs"]:
    print(f"  - {d.metadata['source']}, page {d.metadata['page']} ({d.metadata['status']})")

keys: ['docs', 'question', 'answer']

ANSWER:
You have zero liability for fraudulent transactions reported within 3 days of the card being lost [Credit_Card_Terms_2026.pdf, page 2]. Please report your lost card immediately through the app or by calling 1800-000-1111 to ensure it is blocked [Credit_Card_Terms_2026.pdf, page 2].

SOURCES:
  - Credit_Card_Terms_2026.pdf, page 2 (current)
  - Credit_Card_Terms_2026.pdf, page 1 (current)
  - Credit_Card_Terms_2026.pdf, page 2 (current)
  - Credit_Card_Terms_2026.pdf, page 1 (current)


Streaming works on the whole chain

In [38]:
for piece in rag_chain.stream("How do I escalate a complaint that customer care did not resolve?"):
    print(piece, end="", flush=True)
print()

If you are not satisfied with the resolution from Level 1 (Customer care), you may escalate your complaint to the Grievance Officer [Grievance_Redressal_Policy.pdf, page 1]. The Grievance Officer will provide a response within 15 days [Grievance_Redressal_Policy.pdf, page 1]. If the issue remains unresolved after this level, you may escalate it to the Principal Nodal Officer for a final bank response within 30 days of the original complaint [Grievance_Redressal_Policy.pdf, page 1].
